# Strojenie parametrow udar

In [ ]:
from sklearn.model_selection import GridSearchCV

# Ustalenie danych treningowych na podstawie najlepszej metody balansowania
# Tutaj używamy SMOTE jako przykładu – zmień na najlepszą metodę
best_sampler_name = best_method.split(' (')[0]
print(f'Strojenie modelu z metodą: {best_sampler_name}')

# Dobieramy kernel z najlepszego modelu
best_kernel = 'rbf' if '(rbf)' in best_method else 'linear'

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01] if best_kernel == 'rbf' else ['scale']
}

grid_search = GridSearchCV(
    SVC(kernel=best_kernel, class_weight='balanced', random_state=42),
    param_grid,
    cv=StratifiedKFold(n_splits=5),
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

# Używamy najlepszych zresamplingowanych danych
grid_search.fit(X_smote, y_smote)
print(f'\nNajlepsze parametry: {grid_search.best_params_}')
print(f'Najlepszy Recall (CV): {grid_search.best_score_:.4f}')

y_pred_tuned = grid_search.predict(X_test_scaled)
print(f'Recall na zbiorze testowym: {recall_score(y_test, y_pred_tuned):.4f}')
print(classification_report(y_test, y_pred_tuned))

In [ ]:
# Jeśli model po strojeniu ma lepszy recall, zapisz go zamiast poprzedniego
tuned_recall = recall_score(y_test, y_pred_tuned)
if tuned_recall > best_recall:
    print(f'Model po strojeniu ({tuned_recall:.4f}) lepszy niż poprzedni ({best_recall:.4f}). Aktualizacja...')
    stroke_pred_final = grid_search.predict(X_pred_scaled)
else:
    print(f'Model bazowy ({best_recall:.4f}) lepszy lub równy. Pozostawiamy oryginalne predykcje.')
    stroke_pred_final = stroke_pred

u_csv_final = pd.DataFrame({'stroke': stroke_pred_final})
u_csv_final.to_csv('u.csv', index=False)
print('Finalny plik u.csv zapisany.')
print(f'Rozkład finalnych predykcji: {pd.Series(stroke_pred_final).value_counts().to_dict()}')

# Strojenie parametrow startup

In [ ]:
best_kernel = 'rbf' if '(rbf)' in best_method else 'linear'

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01] if best_kernel == 'rbf' else ['scale']
}

grid_search = GridSearchCV(
    SVC(kernel=best_kernel, class_weight='balanced', random_state=42),
    param_grid,
    cv=StratifiedKFold(n_splits=5),
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_smote, y_smote)

print(f'Najlepsze parametry: {grid_search.best_params_}')
print(f'Najlepszy Recall (CV): {grid_search.best_score_:.4f}')

y_pred_tuned = grid_search.predict(X_test_scaled)
tuned_recall = recall_score(y_test, y_pred_tuned, zero_division=0)
print(f'Recall na zbiorze testowym po strojeniu: {tuned_recall:.4f}')
print(classification_report(y_test, y_pred_tuned))

In [ ]:
if tuned_recall > best_recall:
    print(f'Model po strojeniu ({tuned_recall:.4f}) lepszy. Aktualizacja predykcji...')
    pred_encoded_final = grid_search.predict(X_pred_scaled)
    basari_pred_final = np.where(pred_encoded_final == 1, 'başarılı', 'başarısız')
else:
    print(f'Model bazowy ({best_recall:.4f}) lepszy lub równy.')
    basari_pred_final = basari_pred

s_csv_final = pd.DataFrame({'basari_durumu': basari_pred_final})
s_csv_final.to_csv('s.csv', index=False)
print('Finalny plik s.csv zapisany.')
print(pd.Series(basari_pred_final).value_counts())